In [2]:
import pandas as pd

df = pd.read_csv('100_Unique_QA_Dataset.csv')
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [6]:
# tokenize the dataset: Ie get each unique words

def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'",'')
    return text.split()

In [8]:
tokenize("What is the capital of 'Nepal?")

['what', 'is', 'the', 'capital', 'of', 'nepal']

In [ ]:
# vocabulary creation
vocab={'<UNK>':0}


def build_vocab(row):
    print(row['question'],row['answer']) 
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merged_tokens = tokenized_question + tokenized_answer
    # print(tokenized_question,tokenized_answer)
    # print(merged_tokens)

    for token in merged_tokens:
        if token not in vocab:
            vocab[token]=len(vocab)
    # print(vocab)

In [18]:
print(len(vocab))


324


In [17]:
df.apply(build_vocab, axis=1)

What is the capital of France? Paris
['what', 'is', 'the', 'capital', 'of', 'france', 'paris']
{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'capital': 4, 'of': 5, 'france': 6, 'paris': 7}
What is the capital of Germany? Berlin
['what', 'is', 'the', 'capital', 'of', 'germany', 'berlin']
{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'capital': 4, 'of': 5, 'france': 6, 'paris': 7, 'germany': 8, 'berlin': 9}
Who wrote 'To Kill a Mockingbird'? Harper-Lee
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird', 'harper-lee']
{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'capital': 4, 'of': 5, 'france': 6, 'paris': 7, 'germany': 8, 'berlin': 9, 'who': 10, 'wrote': 11, 'to': 12, 'kill': 13, 'a': 14, 'mockingbird': 15, 'harper-lee': 16}
What is the largest planet in our solar system? Jupiter
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter']
{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'capital': 4, 'of': 5, 'france': 6, 'paris': 7, 'germany': 8, 'berlin': 9, 'who': 10, 'wrot

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [20]:
# convert words to numerical indices 

def text_to_indices(text,vocab):
    indexed_text = []

    for token in tokenize(text):

        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

In [21]:
text_to_indices("who is suman",vocab)

[10, 2, 0]

In [22]:
import torch 
from torch.utils.data import Dataset,DataLoader

In [23]:
class QADataset(Dataset):

    def __init__(self,df,vocab):
        self.df= df
        self.vocab= vocab

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self,index):

        numerical_question =  text_to_indices(self.df.iloc[index]['question'],self.vocab)
        numerical_answer   =  text_to_indices(self.df.iloc[index]['answer'],self.vocab)

        return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [24]:
dataset= QADataset(df,vocab)

In [26]:
dataset[10]

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [61]:
dataloader = DataLoader(dataset,batch_size=1, shuffle=True)

In [28]:
for question,answer in dataLoader:
    print(question,answer)

tensor([[ 42, 299, 300, 118,  14, 301, 302, 158, 303, 304, 305, 306]]) tensor([[307]])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([[246]])
tensor([[  1,   2,   3, 234,   5, 235]]) tensor([[131]])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([[114]])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([[85]])
tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([[254]])
tensor([[ 10, 308,   3, 309, 310]]) tensor([[311]])
tensor([[ 10,  11, 189, 158, 190]]) tensor([[191]])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([[36]])
tensor([[  1,   2,   3,  37,  38,  39, 161]]) tensor([[162]])
tensor([[ 42, 290, 291, 118, 292, 158, 293, 294]]) tensor([[295]])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([[128]])
tensor([[ 1,  2,  3, 69,  5,  3, 70, 71]]) tensor([[72]])
tensor([[  1,   2,   3, 180, 181, 182, 183]]) tensor([[184]])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([[16]])
tensor([[ 42,  18, 118,   3, 186, 187]]) tensor([[188]])
tensor([[  1,   2,   3, 103,   5, 104,  19, 1

RNN Building: 
1 input layer : 50 neurons (embeddings is chosen to be 50 dimensions)
1 hiddern layer : 64 neurons
1 output layer with : 324 neurons (no of total words in vocabulary)

In [30]:
import torch.nn as nn

In [65]:
class SimpleRNN(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=50)

        self.rnn = nn.RNN(50,64,batch_first=True)  # this is necessary

        self.fc  = nn.Linear(64,vocab_size) # squeeze is necessary

    def forward(self,question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)

        output = self.fc(final.squeeze(0))

        return output

What each layer does 

In [37]:
x = nn.Embedding(324,embedding_dim=50)
a = x(dataset[0][0]) # so this embedding layer is converting to 50 dimensions tensor
a

tensor([[ 2.4785,  2.6884, -0.1604,  0.9913, -0.7151, -0.0771, -0.2187, -1.2478,
          0.0137, -0.8003, -0.8205, -0.2195,  1.1564, -0.3202,  1.5313,  1.0903,
         -0.1577,  0.7467,  0.1676,  0.3458,  0.2805,  1.8482, -0.8989,  1.0089,
          0.5278, -2.0504, -0.0288,  0.1683,  0.3815, -0.0081, -0.5870,  0.3656,
         -0.5039, -0.0437, -0.7513, -1.0012,  0.6300,  0.8651,  1.0430, -0.6424,
          0.9001,  1.8951,  0.6709, -0.6490, -0.3154, -0.4250,  2.7180,  0.3365,
          0.2969, -0.6635],
        [-0.6246, -0.5813,  1.5292, -0.2589,  1.0746, -0.9362,  0.3534,  1.2470,
         -2.0642,  1.1649,  1.0040, -1.0370, -1.5402, -0.8815,  0.5109,  1.0225,
         -0.0927,  1.7698,  0.5418,  0.8280, -0.9377, -0.2572,  0.9490, -1.3402,
         -0.9069,  2.4340,  0.2687,  1.9205, -0.4061,  0.9970,  1.3257, -0.0443,
         -0.1350,  2.3531, -1.4217, -0.7734, -1.2593, -0.9155,  0.8539, -1.1781,
          1.3440,  1.2319,  0.7290,  0.0109, -0.6205,  0.3929,  0.5273, -0.1326,


In [39]:
y = nn.RNN(50,64)



torch.Size([1, 64])

In [40]:
#hidden states for ex in 6 words the output of the first to second last ouput will be shown by this 
y(a)[0]

tensor([[-0.5747,  0.7877,  0.2202, -0.3105,  0.4029,  0.5056, -0.0551, -0.2775,
         -0.0609, -0.6352,  0.3000,  0.4655, -0.5332, -0.0019, -0.5622,  0.1840,
          0.4999, -0.5899, -0.2317,  0.3549,  0.5527,  0.0311, -0.4217, -0.2450,
         -0.2791,  0.3477, -0.4938, -0.5974, -0.7979,  0.3972,  0.2555,  0.1044,
          0.5537,  0.1628, -0.1009, -0.6923,  0.1519,  0.6719, -0.6960, -0.5190,
          0.3582,  0.1081,  0.1816,  0.7221,  0.1708, -0.6870, -0.7060,  0.1916,
         -0.3627, -0.1122,  0.3485,  0.3135, -0.8293,  0.7357,  0.6247,  0.3065,
          0.0862,  0.0581,  0.7249, -0.3261, -0.1857, -0.0918, -0.4828,  0.4375],
        [ 0.1386, -0.3491, -0.3286,  0.6408, -0.8435,  0.6258,  0.7686,  0.0121,
          0.6331, -0.1052, -0.5040, -0.3754,  0.5408,  0.6269, -0.5820, -0.0697,
         -0.9165,  0.5277, -0.5659, -0.5094, -0.6177, -0.2283, -0.0361, -0.0454,
          0.8483, -0.3411,  0.0655,  0.6777,  0.2456,  0.4416, -0.7768, -0.6359,
          0.2917,  0.7890, 

In [42]:
# final output 
b= y(a)[1]

In [45]:
z = nn.Linear(64,324)

z(b).shape   # this gives the probability for each word in vocab .

torch.Size([1, 324])

Parameters


In [47]:
learning_rate = 0.001
epochs = 20


In [68]:
model = SimpleRNN(len(vocab))

In [69]:
criterion = nn.CrossEntropyLoss()
optimizer =torch.optim.Adam(model.parameters(), lr=learning_rate)

In [67]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)  # this is necessary batch_first 
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [70]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    # print(output.shape)  
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 524.477468
Epoch: 2, Loss: 454.793374
Epoch: 3, Loss: 376.694966
Epoch: 4, Loss: 316.004927
Epoch: 5, Loss: 264.502905
Epoch: 6, Loss: 215.410109
Epoch: 7, Loss: 170.976885
Epoch: 8, Loss: 131.789130
Epoch: 9, Loss: 100.376954
Epoch: 10, Loss: 76.130845
Epoch: 11, Loss: 58.572170
Epoch: 12, Loss: 45.450585
Epoch: 13, Loss: 36.258941
Epoch: 14, Loss: 29.372808
Epoch: 15, Loss: 24.195544
Epoch: 16, Loss: 20.406907
Epoch: 17, Loss: 17.106656
Epoch: 18, Loss: 14.768522
Epoch: 19, Loss: 12.813242
Epoch: 20, Loss: 11.092095


In [102]:
# predictions 

def predict(model,question,threshold=0.5):

    # convert questions to numbers
    numerical_question = text_to_indices(question,vocab)
    # print(numerical_question)

    #tensor
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)
    # print(question_tensor.shape)

    # sent to model

    output = model(question_tensor)

    # convert logits to probs

    probs = torch.nn.functional.softmax(output,dim=1)
    # print(probs)

    # find index of max probs
    value, index = torch.max(probs, dim=1)
    print(value,index)

    if value < threshold:
        print("I don't Know.")
    # for better confidence then print prediction
    else:
        print(list(vocab.keys())[index])

In [103]:
predict(model,"Who am I ?")

tensor([0.1027], grad_fn=<MaxBackward0>) tensor([112])
I don't Know.


In [108]:
vocab.keys()

dict_keys(['<UNK>', 'what', 'is', 'the', 'capital', 'of', 'france', 'paris', 'germany', 'berlin', 'who', 'wrote', 'to', 'kill', 'a', 'mockingbird', 'harper-lee', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter', 'boiling', 'point', 'water', 'celsius', '100', 'painted', 'mona', 'lisa', 'leonardo-da-vinci', 'square', 'root', '64', '8', 'chemical', 'symbol', 'for', 'gold', 'au', 'which', 'year', 'did', 'world', 'war', 'ii', 'end', '1945', 'longest', 'river', 'nile', 'japan', 'tokyo', 'developed', 'theory', 'relativity', 'albert-einstein', 'freezing', 'fahrenheit', '32', 'known', 'as', 'red', 'mars', 'author', '1984', 'george-orwell', 'currency', 'united', 'kingdom', 'pound', 'india', 'delhi', 'discovered', 'gravity', 'newton', 'how', 'many', 'continents', 'are', 'there', 'on', 'earth', '7', 'gas', 'do', 'plants', 'use', 'photosynthesis', 'co2', 'smallest', 'prime', 'number', '2', 'invented', 'telephone', 'alexander-graham-bell', 'australia', 'canberra', 'ocean', 'pacific-oce

In [ ]:
list(vocab.keys())[112]  # so model has predicted the alexander-fleming internally but it was of less confidence so it gave the I don't know 

'alexander-fleming'

In [104]:
predict(model,"Who discoverd gravity ?")

tensor([0.5340], grad_fn=<MaxBackward0>) tensor([77])
newton


In [106]:
predict(model,"Which year did World War II end?")

tensor([0.9331], grad_fn=<MaxBackward0>) tensor([49])
1945


In [109]:
predict(model,"Which neural network work best for the text generation ?")

tensor([0.0737], grad_fn=<MaxBackward0>) tensor([254])
I don't Know.
